# Getting Started with Linked Data

This notebook shows how to enrich HDF5 metadata with semantic identifiers and export it as RDF.

With `h5rdmtoolbox`, HDF5 objects and attributes can be interpreted as RDF triples:

- the HDF5 object becomes the **subject**
- the attribute name becomes the **predicate**
- the attribute value becomes the **object**

This makes HDF5 metadata machine-interpretable and easier to reuse in FAIR data workflows.

## Why linked data for HDF5?

Standard HDF5 attributes are useful for humans, but often ambiguous for machines.
For example, an attribute like `"units": "degree_Celsius"` is readable, but it does not
by itself identify a shared concept from a controlled vocabulary.

By attaching [IRI](https://en.wikipedia.org/wiki/Internationalized_Resource_Identifier)s to attribute names and values, we can make the meaning explicit.
This allows HDF5 metadata to be exported as RDF (as serialization in e.h. JSON-LD or Turtle) and used in
semantic web and FAIR workflows.


The `ld` module provides:
- Conversion of HDF5 files to RDF graphs
- Export to JSON-LD and Turtle formats
- SHACL validation (see [shacl_validation](shacl_validation.ipynb))
- Semantic mapping of attributes to ontologies

In [ ]:
import numpy as np
import rdflib
import h5rdmtoolbox as h5tbx
from pathlib import Path

## Create an HDF5 file with semantic annotations

In this example, we create a small HDF5 file containing temperature and pressure data.
We annotate:

- the dataset description using `schema.org`
- the unit using Metadata4Ing and a QUDT unit IRI
- the dataset type and data predicate for a derived quantity

In [ ]:
M4I = rdflib.Namespace("http://w3id.org/nfdi4ing/metadata4ing#")

hdf_filename = "linked_data_example.h5"

with h5tbx.File(hdf_filename, "w") as h5:
    ds = h5.create_dataset("temperature", data=np.array([20.0, 21.0, 19.0, 22.0]))

    ds.attrs["units"] = h5tbx.Attribute(
        value="degree_Celsius",
        rdf_predicate=M4I.hasUnit,
        rdf_object="http://qudt.org/vocab/unit/DEG_C",
    )
    ds.attrs["description", "https://schema.org/description"] = "Room temperature measurements"

    ds_mean = h5.create_dataset("mean_temperature", data=np.mean(ds[()]))
    ds_mean.attrs["units", M4I.hasUnit] = "degree_Celsius"
    ds_mean.rdf["units"].object = "http://qudt.org/vocab/unit/DEG_C"
    ds_mean.attrs["description", "https://schema.org/description"] = "Mean room temperature"

    ds_mean.rdf.type = M4I.NumericalVariable
    ds_mean.rdf.data_predicate = M4I.hasNumericalValue

## Inspect the file

Before exporting RDF, it helps to inspect the HDF5 content and confirm that the
semantic annotations were written as expected.

In [ ]:
h5tbx.dump(hdf_filename, collapsed=False)

## Export RDF as Turtle

`h5rdmtoolbox` can serialize HDF5 metadata as RDF. A useful first format is Turtle,
because it is compact and readable.

In [ ]:
ttl = h5tbx.serialize(
    hdf_filename,
    format="ttl",
    structural=False,
    semantic=True,
    indent=2,
    context={
        "m4i": "http://w3id.org/nfdi4ing/metadata4ing#",
        "schema": "https://schema.org/",
    },
)

print(ttl)

## Export RDF as JSON-LD

JSON-LD is often the easiest RDF serialization to exchange with web-based tools and
downstream FAIR workflows.

In [ ]:
jsonld = h5tbx.serialize(
    hdf_filename,
    format="json-ld",
    structural=False,
    semantic=True,
    indent=2,
    context={
        "m4i": "http://w3id.org/nfdi4ing/metadata4ing#",
        "schema": "https://schema.org/",
    },
)

print(jsonld[:2000])  # print only the first part to keep the notebook compact

## Parse the exported RDF with `rdflib`

Once exported, the metadata can be loaded into a standard RDF library and queried.

In [ ]:
from rdflib import Graph

g = Graph()
g.parse(data=jsonld, format="json-ld")

print(f"Number of triples: {len(g)}")

## Query the graph

We can now ask semantic questions about the data, for example: which resources have
a `schema:description` and what is the value?

We will use the built-in method `sparql(...)` (alternatively use `h5tbx.sparql(...)`), which will generate the RDF data and apply the query in the background. For better readability we provide a base file URI and request the return data type to be a pandas dataframe:

In [ ]:
query = """
PREFIX schema: <https://schema.org/>

SELECT ?resource ?description
WHERE {
    ?resource schema:description ?description .
}
"""

res = h5tbx.sparql(hdf_filename, query, as_dataframe=True, file_uri="https://example.org#")
res

## Structural vs semantic RDF

`h5rdmtoolbox` can export:

- **semantic/contextual RDF**: only the semantics you explicitly added
- **structural RDF**: information derived from the HDF5 structure
- **full RDF**: a combination of both

In [ ]:
ttl_semantic = h5tbx.serialize(hdf_filename, format="ttl", structural=False, semantic=True)
ttl_structural = h5tbx.serialize(hdf_filename, format="ttl", structural=True, semantic=False)
ttl_full = h5tbx.serialize(hdf_filename, format="ttl", structural=True, semantic=True)

print("semantic length:", len(ttl_semantic))
print("structural length:", len(ttl_structural))
print("full length:", len(ttl_full))

In [ ]:
print(ttl_semantic[:1000])

In [ ]:
print(ttl_structural[:2000])

In [ ]:
print(ttl_full[:2000])

## Next step: Validate metadata with SHACL

Because the metadata is available as RDF, it can be validated with SHACL.
This is useful when you want to check whether required metadata is present and correctly typed.

A full SHACL example is covered in a [separate notebook](shacl_validation.ipynb)), but the key idea is that you can
define RDF constraints and validate the exported graph against them.